# 04 — Data Cleaning

This notebook creates a separate cleaned dataset for EDA while preserving the integrated Master Dataset unchanged.

In [15]:
# Import libraries
from pathlib import Path
import sys

In [16]:
# set project root and config path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'data_cleaning.yaml'
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/data_cleaning.yaml')

## Review the cleaning configuration

In [17]:
import yaml

with CONFIG_PATH.open('r', encoding='utf-8') as file:
    cleaning_config = yaml.safe_load(file)

cleaning_config

{'paths': {'input_master_dataset': 'data/processed/master_dataset.parquet',
  'output_clean_dataset': 'data/processed/master_dataset_clean.parquet',
  'reports_dir': 'reports/data_cleaning',
  'docs_dir': 'docs'},
 'cleaning': {'columns_to_drop': {'SOURCE_FILE': {'reason': 'Weather source-file traceability is already documented upstream and is not an analytical variable.',
    'rule': 'drop_if_present'},
   'SOURCE_PERIOD': {'reason': 'Monthly source-period traceability is already documented upstream and duplicates information available from timestamp.',
    'rule': 'drop_if_present'},
   'Dew Point Temp Flag': {'reason': 'Near-empty ECCC quality flag; not sufficiently populated for analysis.',
    'rule': 'drop_if_missing_pct_at_least',
    'threshold_pct': 99.9},
   'Rel Hum Flag': {'reason': 'Near-empty ECCC quality flag; not sufficiently populated for analysis.',
    'rule': 'drop_if_missing_pct_at_least',
    'threshold_pct': 99.9},
   'Visibility Flag': {'reason': 'Near-empty ECC

## Run the complete cleaning pipeline

In [18]:
# Import the function to clean the master dataset
from src.ontario_peak_risk.data_cleaning.clean_master_dataset import clean_master_dataset

In [19]:
# Clean the master dataset using the configuration file
clean_dataset = clean_master_dataset(CONFIG_PATH)
clean_dataset.shape

e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\src\ontario_peak_risk\data_cleaning\clean_master_dataset.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = dst.groupby(["fsa", "date_check"])["timestamp"].nunique()


Created cleaned dataset: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\data\processed\master_dataset_clean.parquet
Rows: 262,944
Columns: 66
Reports: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\reports\data_cleaning
Documentation: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\docs\Data_Cleaning_Report.md


(262944, 66)

## Verify final key and period

In [20]:
{
    'rows': len(clean_dataset),
    'columns': clean_dataset.shape[1],
    'fsas': sorted(clean_dataset['fsa'].astype(str).unique().tolist()),
    'minimum_timestamp': clean_dataset['timestamp'].min(),
    'maximum_timestamp': clean_dataset['timestamp'].max(),
    'duplicate_fsa_timestamp_rows': int(
        clean_dataset.duplicated(['fsa', 'timestamp'], keep=False).sum()
    ),
    'missing_target': int(clean_dataset['total_consumption_kwh'].isna().sum()),
}

{'rows': 262944,
 'columns': 66,
 'fsas': ['L4T', 'M5R', 'M5S', 'M6G', 'M9R', 'M9W'],
 'minimum_timestamp': Timestamp('2021-01-01 00:00:00'),
 'maximum_timestamp': Timestamp('2025-12-31 23:00:00'),
 'duplicate_fsa_timestamp_rows': 0,
 'missing_target': 0}

## Review remaining missing values

In [21]:
(
    clean_dataset.isna().mean().mul(100)
    .sort_values(ascending=False)
    .rename('missing_pct')
    .to_frame()
    .query('missing_pct > 0')
)

,missing_pct
Wind Chill,89.980376
Hmdx,82.395491
Weather,78.641840
Wind Dir (10s deg),50.835159
Precip. Amount (mm),50.046778
Visibility (km),50.007986
Wind Spd (km/h),50.006846
Dew Point Temp (°C),0.054765
Rel Hum (%),0.054765
Temp (°C),0.053624


## Review generated reports

In [22]:
import pandas as pd

In [23]:
# View consistency_checks

REPORTS_DIR = PROJECT_ROOT / 'reports' / 'data_cleaning'
consistency_checks = pd.read_csv(REPORTS_DIR / 'consistency_checks.csv')
outlier_review = pd.read_csv(REPORTS_DIR / 'outlier_review.csv')

consistency_checks

,check,violations,severity,status,description
0,required_key_not_null,0,error,PASS,FSA and timestamp must be present for every row.
1,unique_fsa_timestamp,0,error,PASS,The cleaned dataset must contain one row per F...
2,required_non_null::fsa,0,error,PASS,Required field 'fsa' must not be missing.
3,required_non_null::timestamp,0,error,PASS,Required field 'timestamp' must not be missing.
4,required_non_null::total_consumption_kwh,0,error,PASS,Required field 'total_consumption_kwh' must no...
5,required_non_null::reported_premise_count,0,error,PASS,Required field 'reported_premise_count' must n...
6,non_negative_consumption,0,error,PASS,Electricity consumption must not be negative.
7,non_negative_premise_count,0,error,PASS,Reported premise count must not be negative.
8,relative_humidity_range,0,error,PASS,Relative humidity must be between 0 and 100 pe...
9,non_negative::Precip. Amount (mm),0,error,PASS,'Precip. Amount (mm)' must not contain negativ...


In [24]:
# Review the outlier review report
outlier_review

,column,non_null_count,minimum,p01,q1,median,q3,p99,maximum,iqr_lower_bound,iqr_upper_bound,iqr_outlier_count,iqr_outlier_pct,action
0,total_consumption_kwh,262944,1377.7,2586.329,5224.90,8111.40,11544.00,21178.17,33712.20,-4253.75,21022.65,2768,1.052696,reported_only_not_removed
1,reported_premise_count,262944,5803.0,5873.000,6024.00,8674.00,11176.00,12099.00,12120.00,-1704.00,18904.00,0,0.000000,reported_only_not_removed
2,Temp (°C),262803,-21.8,-11.900,1.90,10.10,19.20,29.40,35.80,-24.05,45.15,0,0.000000,reported_only_not_removed
3,Dew Point Temp (°C),262800,-30.1,-17.900,-3.20,3.90,12.60,21.50,26.00,-26.90,36.30,108,0.041096,reported_only_not_removed
4,Rel Hum (%),262800,14.0,28.000,56.00,68.00,80.00,100.00,100.00,20.00,116.00,165,0.062785,reported_only_not_removed
5,Precip. Amount (mm),131349,0.0,0.000,0.00,0.00,0.00,2.30,45.00,0.00,0.00,10791,8.215517,reported_only_not_removed
6,Wind Dir (10s deg),129276,1.0,1.000,14.00,24.00,30.00,36.00,36.00,-10.00,54.00,0,0.000000,reported_only_not_removed
7,Wind Spd (km/h),131454,0.0,0.000,9.00,14.00,21.00,43.00,67.00,-9.00,39.00,2520,1.917020,reported_only_not_removed
8,Visibility (km),131451,0.0,1.600,24.10,24.10,24.10,24.10,80.50,24.10,24.10,21933,16.685305,reported_only_not_removed
9,Stn Press (kPa),262803,95.7,97.700,99.31,99.88,100.43,101.72,103.25,97.63,102.11,2802,1.066198,reported_only_not_removed


## Completion criterion

Proceed to EDA only when all hard consistency checks show `PASS`. Do not delete reported demand outliers.